# Gender Wage Gap in North Carolina

This notebook downloads 2024 ACS 1-Year PUMS data from the Census API, cleans it, and creates weighted descriptive charts. Run every cell in order. Do not commit a real API key.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import seaborn as sns

CENSUS_API_KEY = 'PASTE_YOUR_KEY_HERE'  # Do not save a real key in GitHub.
BASE_URL = 'https://api.census.gov/data/2024/acs/acs1/pums'
FIELDS = ['AGEP', 'SEX', 'WAGP', 'SCHL', 'WKHP', 'WKWN', 'PWGTP']
ROOT = Path('..') if Path('..').joinpath('README.md').exists() else Path('.')
DATA_DIR = ROOT / 'data'
IMAGE_DIR = ROOT / 'assets' / 'images'
DATA_DIR.mkdir(exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
assert CENSUS_API_KEY != 'PASTE_YOUR_KEY_HERE', 'Paste your Census API key above before running.'

In [ ]:
params = {'get': ','.join(FIELDS), 'for': 'state:37', 'key': CENSUS_API_KEY}
response = requests.get(BASE_URL, params=params, timeout=60)
response.raise_for_status()
payload = response.json()
raw = pd.DataFrame(payload[1:], columns=payload[0])
raw.head()

In [ ]:
numeric_columns = ['AGEP', 'SEX', 'WAGP', 'SCHL', 'WKHP', 'WKWN', 'PWGTP']
for column in numeric_columns:
    raw[column] = pd.to_numeric(raw[column], errors='coerce')

# Working-age adults with positive wage/salary income and valid fields.
df = raw.loc[
    raw['AGEP'].between(25, 64)
    & raw['WAGP'].gt(0)
    & raw['SEX'].isin([1, 2])
    & raw['SCHL'].between(1, 24)
    & raw['WKHP'].between(1, 98)
    & raw['WKWN'].between(1, 52)
    & raw['PWGTP'].gt(0)
].copy()

df['sex_label'] = df['SEX'].map({1: 'Men', 2: 'Women'})
def education_group(code):
    if code <= 15: return 'High school or less'
    if code <= 19: return 'Some college or associate degree'
    if code == 20: return 'Bachelor’s degree'
    return 'Graduate or professional degree'
df['education_group'] = df['SCHL'].apply(education_group)
df.to_csv(DATA_DIR / 'cleaned_north_carolina_wages.csv', index=False)
print(f'Rows kept: {len(df):,}')
df[['AGEP', 'WAGP', 'sex_label', 'education_group']].describe(include='all')

In [ ]:
def weighted_median(values, weights):
    order = np.argsort(values)
    values, weights = np.asarray(values)[order], np.asarray(weights)[order]
    return values[np.cumsum(weights).searchsorted(weights.sum() / 2)]

def summary(group):
    return pd.Series({
        'weighted_median_wage': weighted_median(group['WAGP'], group['PWGTP']),
        'weighted_mean_wage': np.average(group['WAGP'], weights=group['PWGTP']),
        'unweighted_records': len(group),
    })

by_sex = df.groupby('sex_label', observed=True).apply(summary, include_groups=False).reset_index()
by_education = df.groupby(['education_group', 'sex_label'], observed=True).apply(summary, include_groups=False).reset_index()
display(by_sex)
display(by_education)

In [ ]:
sns.set_theme(style='whitegrid')
fig, ax = plt.subplots(figsize=(7, 5))
sns.barplot(data=by_sex, x='sex_label', y='weighted_median_wage', hue='sex_label', legend=False, ax=ax)
ax.set(title='Weighted Median Annual Wage/Salary Income by Sex', xlabel='', ylabel='2024 dollars')
ax.yaxis.set_major_formatter(lambda value, position: f'${value:,.0f}')
fig.tight_layout()
fig.savefig(IMAGE_DIR / 'weighted_median_wages_by_sex.png', dpi=200)
plt.show()

order = ['High school or less', 'Some college or associate degree', 'Bachelor’s degree', 'Graduate or professional degree']
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=by_education, x='education_group', y='weighted_median_wage', hue='sex_label', order=order, ax=ax)
ax.set(title='Weighted Median Annual Wage/Salary Income by Education and Sex', xlabel='', ylabel='2024 dollars')
ax.tick_params(axis='x', rotation=18)
ax.yaxis.set_major_formatter(lambda value, position: f'${value:,.0f}')
fig.tight_layout()
fig.savefig(IMAGE_DIR / 'weighted_median_wages_by_education.png', dpi=200)
plt.show()

## Before publishing

Read the tables and charts, then replace the bracketed findings in `gender-wage-gap.md` with your exact results. Keep the limitations section: these comparisons are descriptive and do not establish a cause of wage differences.